In [3]:
# Cell 1: Environment Setup and Imports
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
!{sys.executable} -m pip install wandb torchmetrics seaborn
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data.sampler import WeightedRandomSampler
import matplotlib.pyplot as plt


# Add parent directory to path to import custom modules
parent_dir = r'c:\Users\dulne\OneDrive\Desktop\IIT\4th year\FYP\Class Imbalance For lesion and Dermatology\implementation\ReLief'
sys.path.append(parent_dir)

import sys
print(sys.executable)

import models
import lesion_datasets
import train

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Environment setup complete")

c:\Users\dulne\anaconda3\python.exe
Environment setup complete


In [4]:
# Cell 2: Configuration and Paths
# Dataset Configuration
DATA_DIR = r'c:\Users\dulne\OneDrive\Desktop\IIT\4th year\FYP\Class Imbalance For lesion and Dermatology\implementation\ReLief\data\img'
IMG_CLASS_NAMES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

# Training Hyperparameters (from the best experiment)
NUM_EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 0.0005
AUGMENT = True
FIVE_CROP = False

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [5]:
# Cell 3: Load Dataset Labels
# Load training and validation labels
train_df = pd.read_csv(f'{DATA_DIR}\\train.csv')
val_df = pd.read_csv(f'{DATA_DIR}\\val.csv')

# Display dataset information
print("Dataset Statistics:")
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"\nClass distribution in training set:")
train_labels = train_df[IMG_CLASS_NAMES].values.argmax(axis=1)
for i, class_name in enumerate(IMG_CLASS_NAMES):
    count = np.sum(train_labels == i)
    print(f"{class_name}: {count} ({count/len(train_labels)*100:.1f}%)")

Dataset Statistics:
Training samples: 3004
Validation samples: 400

Class distribution in training set:
MEL: 267 (8.9%)
NV: 2052 (68.3%)
BCC: 151 (5.0%)
AKIEC: 114 (3.8%)
BKL: 339 (11.3%)
DF: 36 (1.2%)
VASC: 45 (1.5%)


In [6]:
# Cell 4: Configure Weighted Sampling for Class Imbalance
print("Configuring weighted sampling for class imbalance...")

# Calculate class frequencies
train_labels_for_sampling = train_df[IMG_CLASS_NAMES].values.argmax(axis=1)
class_sample_count = np.array([
    np.sum(train_labels_for_sampling == t) for t in range(len(IMG_CLASS_NAMES))
])

# Compute inverse frequency weights
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in train_labels_for_sampling])
samples_weight = torch.from_numpy(samples_weight).float()

# Create weighted sampler
sampler = WeightedRandomSampler(
    samples_weight, 
    len(samples_weight), 
    replacement=True
)

print(f"Class distribution: {class_sample_count}")
print(f"Class weights: {weight}")

Configuring weighted sampling for class imbalance...
Class distribution: [ 267 2052  151  114  339   36   45]
Class weights: [0.00374532 0.00048733 0.00662252 0.00877193 0.00294985 0.02777778
 0.02222222]


In [7]:
# Cell 5: Create Datasets and DataLoaders
# Training dataset with augmentation
train_dataset = lesion_datasets.LesionDataset(
    DATA_DIR, 
    f'{DATA_DIR}\\train.csv', 
    augment=AUGMENT,
    five_crop=FIVE_CROP
)

# Validation dataset without augmentation
val_dataset = lesion_datasets.LesionDataset(
    DATA_DIR, 
    f'{DATA_DIR}\\val.csv', 
    augment=False,
    five_crop=FIVE_CROP
)

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    sampler=sampler,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    num_workers=2
)

print(f"Training batches per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Samples per training batch: {BATCH_SIZE}")

Training batches per epoch: 47
Validation batches: 7
Samples per training batch: 64


In [8]:
import lesion_datasets
print(f"lesion_datasets module location: {lesion_datasets.__file__}")
print(f"Available attributes: {dir(lesion_datasets)}")

lesion_datasets module location: c:\Users\dulne\OneDrive\Desktop\IIT\4th year\FYP\Class Imbalance For lesion and Dermatology\implementation\ReLief\lesion_datasets.py
Available attributes: ['Image', 'LesionDataset', 'Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'collections', 'csv', 'np', 'pd', 'torch', 'transforms']


In [9]:
# Cell 6: Model Initialization
print("Initializing model...")

# Initialize ResNet18 with pre-trained weights
model = models.get_pretrained_resnet18(
    num_classes=len(IMG_CLASS_NAMES), 
    freeze_features=False  # Allow fine-tuning of all layers
)

# Move model to device
model = model.to(device)

# Initialize optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

print(f"Model: ResNet18 (pre-trained)")
print(f"Number of classes: {len(IMG_CLASS_NAMES)}")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")
print(f"Loss function: CrossEntropyLoss")
print(f"Device: {device}")

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Initializing model...
Model: ResNet18 (pre-trained)
Number of classes: 7
Optimizer: Adam (lr=0.0005)
Loss function: CrossEntropyLoss
Device: cpu

Total parameters: 11,180,103
Trainable parameters: 11,180,103


In [ ]:
# Cell 7: Train the Model
print("Starting training...")

# Initialize Weights & Biases (optional - you can skip this if you don't want to use wandb)
import wandb
import os

# Login to wandb (you'll need to enter your API key the first time)

os.environ["WANDB_MODE"] = "offline"
wandb.init(project="Skin-Lesion-Baseline", mode="offline")


#Start training with the train.train_model function
# Start training with the train.train_model function
train.train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    class_names=IMG_CLASS_NAMES,
    n_epochs=NUM_EPOCHS, 
    project_name="Skin-Lesion-Baseline",
    ident_str=f"resnet18_aug{AUGMENT}_lr{LEARNING_RATE}_bs{BATCH_SIZE}_sampler"
)


print("\nTraining complete!")

Starting training...


Epochs:   0%|          | 0/5 [00:00<?, ?it/s]